In [10]:
import pandas as pd
from pathlib import Path

In [11]:
data_path = Path("../data")

In [12]:
# This finds all folders inside data/ and its subfolders
folders = [f for f in data_path.rglob("*") if f.is_dir()]
print("Total recording folders found:", len(folders))

Total recording folders found: 106


In [14]:
all_data = []

for folder in folders:

    # Only process folders that contain accelerometer.csv
    if (folder / "accelerometer.csv").exists() and (folder / "gyroscope.csv").exists():

        accel = pd.read_csv(folder / "accelerometer.csv")
        gyro = pd.read_csv(folder / "gyroscope.csv")

        # rename columns to avoid clash
        accel = accel.rename(columns={"x": "accel_x", "y": "accel_y", "z": "accel_z"})
        gyro = gyro.rename(columns={"x": "gyro_x", "y": "gyro_y", "z": "gyro_z"})

        # merge accelerometer + gyroscope using nearest timestamp
        accel = accel.sort_values("seconds_elapsed")
        gyro = gyro.sort_values("seconds_elapsed")

        df = pd.merge_asof(accel, gyro, on="seconds_elapsed", direction="nearest")

        df["recording"] = folder.name  # keep track of which recording

        all_data.append(df)

In [15]:
full_dataset = pd.concat(all_data, ignore_index=True)
print("Combined dataset shape:", full_dataset.shape)

Combined dataset shape: (97784, 10)


In [16]:
full_dataset.head()
full_dataset.to_csv("../processed_data/merged_sensor_data.csv", index=False)

In [17]:
activity_map = {}

for folder in full_dataset['recording'].unique():
    if 'walking' in folder.lower():
        activity_map[folder] = 'walking'
    elif 'standing' in folder.lower():
        activity_map[folder] = 'standing'
    elif 'jumping' in folder.lower():
        activity_map[folder] = 'jumping'
    elif 'still' in folder.lower():
        activity_map[folder] = 'still'
    else:
        activity_map[folder] = 'unknown'

In [18]:
full_dataset['activity'] = full_dataset['recording'].map(activity_map)

In [19]:
full_dataset['activity'].value_counts()

activity
standing    25781
walking     24627
still       23819
jumping     22500
unknown      1057
Name: count, dtype: int64

In [20]:
for recording_id, group in full_dataset.groupby("recording"):
    # group is one recording
    # create windows and extract features from this group
    pass